# Step 10: per-slice normalisation for the ViT + ComBat on ViT features (GPU, ~4-5 h)
The professor's "normalise each slice separately", tested on the deep models. Each per-slice method has a
**volume twin**, so each pair differs only in where the statistics come from:

| per slice (new) | volume twin | what it does |
|---|---|---|
| `slice_joint` | `joint_wide` | one 0.5/99.9 pct window for the 3 phases |
| `slice_pre` | `pre_ref` | log(1 + x / r), r = mean pre-contrast foreground |
| `slice_zscore` | `zscore` | z-score by the pre-contrast foreground |
| `slice_phase` | `per_phase` | each phase its own window: destroys the enhancement, ablation only |

1. **Probe** (frozen DINOv2 + MAE, ~30 min): every norm above, CV AUC on train + val AUC; per-slice vs twin.
2. **ComBat on the ViT features** (~10 min): the probe's patient embeddings, with and without ComBat and with
   and without subtype (same scoring as step 9). Attach the step-9 output to also get "ViT + 3d+temporal".
3. **Fine-tuning** (~0.4 h per run): DINOv2, 2 trainable blocks (Akula's best), each norm in `FT_NORMS` x
   `FT_SEEDS`. Reports the best-epoch val AUC **and the mean val AUC over epochs 5-15** (the honest number;
   the peak epoch is optimistic), per cohort (is Duke still at chance?) and paired tests per-slice vs twin.

Kaggle setup: Accelerator **GPU T4 x2**, Internet **ON**.
**+ Add Input** (any cache works; the best one attached per cohort is used and printed):
- the caches, best first: **`mnj-cache2-ispy1/-ispy2/-duke`** (raw, step 2: nothing clipped, exact features), or
  Akula's **`mnj-cache-v2`** (uint8, 0.7 mm, less clipping), or `mnj-cache-ispy1/-ispy2/-duke` (uint8, clipped);
- the code: the `mnj-code-vaib` Dataset (upload the new zip as a **new version** first) or the `GITHUB_TOKEN` secret.

Run with `DEBUG = True` first (a few minutes, 40 patients per split) to check the notebook, then `DEBUG = False`
with **Save Version -> Save & Run All (Commit)**. Every step prints a timestamped `[...]` line, so the log shows
where it is and which patient or configuration failed. Results: Output tab, `results/<PERSON>/<step>/`.
Optional: **+ Add Input -> Your Work -> Notebooks -> your step-9 notebook** (for ViT + hand-crafted).

In [ ]:
# ==== PARAMETERS (edit these) ====
REPO = "AkulaReddy/MRI_chemo_response"
GIT_TAG = "vaib"           # branch (or tag) to clone
PERSON = ""               # optional: "A", "B" or "C" -> results/<PERSON>/ (empty -> results/vaib/)
STEP = "step10"
PREFER = "v2"
RUN_PROBE = True
PROBE_BACKBONES = ["dinov2_b", "mae"]
PROBE_NORMS = ["joint_wide", "slice_joint", "pre_ref", "slice_pre", "zscore", "slice_zscore", "per_phase",
               "slice_phase", "early3", "kinetic"]
PROBE_CROPS = ["tumor"]
RUN_DEEP_COMBAT = True       # ComBat / subtype on the probe's embeddings
RUN_FT = True
FT_CONFIG = "vo_dinov2"      # DINOv2-B/14 (configs/vo_dinov2.yaml)
FT_NORMS = ["pre_ref", "slice_pre", "joint_wide", "slice_joint"]   # twins next to each other
FT_SEEDS = [0, 1]
FT_EXTRA = ["model.train_blocks=2", "data.crop=tumor"]           # Akula's best: 2 trainable blocks
TWINS = {"slice_joint": "joint_wide", "slice_pre": "pre_ref", "slice_zscore": "zscore", "slice_phase": "per_phase"}
TIME_BUDGET_H = 11.0
DEBUG = False                # True: probe on 40 patients, fine-tuning 2 epochs on 48 patients
PUSH_RESULTS = False
NOTEBOOK_NAME = "step10"

In [ ]:
# ==== SETUP (do not edit): get the repo code, install requirements ====
import base64, json, os, shutil, subprocess, sys
if PERSON not in ("A", "B", "C"):           # only names the results folder
    PERSON = "vaib"
    print('PERSON not set: results go to results/vaib/ (set PERSON = "A", "B" or "C" to change)')
_tok = None
try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception as e:
    print("no GITHUB_TOKEN secret attached to this notebook (fine if the mnj-code-vaib Dataset is attached):",
          type(e).__name__)

def git(*args, cwd="/tmp/repo"):
    """git with the token as a one-off header (never stored in .git/config or the output)."""
    if not _tok:
        raise RuntimeError("no GITHUB_TOKEN secret: git push/pull is not available")
    hdr = "Authorization: Basic " + base64.b64encode(f"x-access-token:{_tok}".encode()).decode()
    r = subprocess.run(["git", "-c", f"http.extraHeader={hdr}", *args], cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(_tok, "***")
    if r.returncode:
        raise RuntimeError(f"git {args[0]} failed:\n{out}")
    return out

shutil.rmtree("/tmp/repo", ignore_errors=True)
if _tok:                                    # 1) clone the branch from GitHub
    git("clone", "--quiet", "--depth", "1", "--branch", GIT_TAG, f"https://github.com/{REPO}.git", "/tmp/repo", cwd="/tmp")
    print("code: GitHub", REPO, GIT_TAG, git("log", "-1", "--format=%h %s"))
else:                                       # 2) or a code Dataset (zip of the repo) attached as input
    import glob, zipfile
    hits = [h for d in range(1, 8) for h in glob.glob("/kaggle/input/" + "*/" * d + "src/vitonly/__init__.py")]
    zips = [z for d in range(1, 6) for z in glob.glob("/kaggle/input/" + "*/" * d + "mnj-code-vaib*.zip")]
    if not hits and zips:                   # Kaggle kept the uploaded zip as a file: unpack it
        zipfile.ZipFile(zips[0]).extractall("/tmp/code_zip")
        hits = glob.glob("/tmp/code_zip/**/src/vitonly/__init__.py", recursive=True)
    if not hits:
        print("attached inputs:")
        for root, dirs, files in os.walk("/kaggle/input"):
            if root.count("/") <= 5:
                print("  ", root, dirs[:6], files[:4])
            else:
                dirs[:] = []
        raise SystemExit(
            "\nThe notebook can't find the project code. Do ONE of these, then Run All again:\n"
            "  (a) Add-ons -> Secrets -> add GITHUB_TOKEN (a GitHub token with access to the repo) and switch it ON\n"
            "      for this notebook, or\n"
            "  (b) + Add Input -> Datasets -> Your Datasets -> mnj-code-vaib (made by uploading mnj-code-vaib.zip).")
    code_root = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))
    shutil.copytree(code_root, "/tmp/repo")
    print("code: attached Dataset", code_root)
os.chdir("/tmp/repo")
sys.path.insert(0, "/tmp/repo")         # so `from src...` works in this kernel
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)
import torch; print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
# ==== HELPERS (do not edit): training runs, prediction files, comparisons ====
import glob, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src import evaluation as ev, stats
from src.train import find_caches
from src.data.metadata import patient_table
COH = ["ispy1", "ispy2", "duke"]
OUT = f"/kaggle/working/results/{PERSON}/{STEP}"
os.makedirs(OUT, exist_ok=True)
T0 = time.time()
RUNS_JSON = {}
pd.set_option("display.width", 250, "display.max_columns", 40)

def budget_left():
    return TIME_BUDGET_H - (time.time() - T0) / 3600

def say(msg):
    """Timestamped progress line (elapsed and remaining budget), flushed so it shows in the Kaggle log at once."""
    el = (time.time() - T0) / 3600
    print(f"[{time.strftime('%H:%M:%S')} | {el:.2f} h elapsed, {budget_left():.2f} h left] {msg}", flush=True)

def run_cmd(cmd):
    """Run a training subprocess and stream its output line by line into this notebook (subprocess.run alone
    shows nothing until it ends, and the child's stdout is block-buffered without PYTHONUNBUFFERED)."""
    print("\n" + " ".join(cmd), flush=True)
    t = time.time()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise subprocess.CalledProcessError(p.returncode, cmd)
    say(f"finished in {(time.time() - t) / 60:.1f} min")

def check_caches(version):
    caches = find_caches(["/kaggle/input", "/kaggle/working"])
    info = {c: json.load(open(f"{d}/cache_meta.json")).get("builder_version", 1) for c, d in caches.items()}
    print("caches:", {c: f"builder v{v}: {caches[c]}" for c, v in info.items()})
    if version is not None:
        bad = {c: v for c, v in info.items() if v != version}
        assert not bad and len(info) == 3, (f"need cache builder v{version} for all 3 cohorts (attach mnj-cache-v2); "
                                            f"found {info}")
    return caches

def _copy(d, label):
    os.makedirs(f"{OUT}/{label}", exist_ok=True)
    for f in ("val_preds.csv", "metrics.json", "log.csv", "config_resolved.yaml", "export_preds.csv",
              "train_preds.csv", "encoder.json"):
        if os.path.exists(f"{d}/{f}"):
            shutil.copy(f"{d}/{f}", f"{OUT}/{label}/{f}")

def train(config, seed, sets=(), label=None, cohorts=None):
    """One src.train run (B3 / F2 ...) -> (run_dir, metrics). Its small files are copied to OUT/<label>/."""
    cmd = [sys.executable, "-m", "src.train", "--config", config, "--seed", str(seed), "--out", "/kaggle/working/runs"]
    if CACHE_VERSION is not None: cmd += ["--cache-version", str(CACHE_VERSION)]
    if cohorts: cmd += ["--cohorts", *cohorts]
    if DEBUG: cmd.append("--debug")
    if sets: cmd += ["--set", *sets]
    run_cmd(cmd)
    d = max(glob.glob("/kaggle/working/runs/*/"), key=os.path.getmtime).rstrip("/")
    if label: _copy(d, label)
    return d, json.load(open(f"{d}/metrics.json"))

def finetune(config, seed, sets=(), label=None, cohorts=None):
    """One src.vitonly.finetune run (our pipeline: ViT-only vo_base / fusion vo_fusion) -> run_dir."""
    cmd = [sys.executable, "-m", "src.vitonly.finetune", "--config", config, "--seed", str(seed),
           "--cache-root", "/kaggle/input", "/kaggle/working", "--out", "/kaggle/working/runs"]
    if cohorts: cmd += ["--cohorts", *cohorts]
    if DEBUG: cmd.append("--debug")
    if sets: cmd += ["--set", *sets]
    run_cmd(cmd)
    d = max(glob.glob("/kaggle/working/runs/*/"), key=os.path.getmtime).rstrip("/")
    if label: _copy(d, label)
    return d

def note_runs(key, dirs):
    """label -> run folder names, in OUT/runs.json: step 8 uses it to find the checkpoints."""
    RUNS_JSON[key] = [os.path.basename(d) for d in dirs]
    json.dump(RUNS_JSON, open(f"{OUT}/runs.json", "w"), indent=1)

def preds_of(dirs):
    fr = [ev.load_preds(f"{d}/val_preds.csv") for d in dirs]
    return fr[0] if len(fr) == 1 else ev.ensemble(fr)

def save_preds(label, df):
    os.makedirs(f"{OUT}/{label}", exist_ok=True)
    df.to_csv(f"{OUT}/{label}/val_preds.csv", index=False)
    return df

def repo_preds(pattern):
    hits = sorted(glob.glob(f"results/*/{pattern}/val_preds.csv"))
    return preds_of([os.path.dirname(h) for h in hits]) if hits else None

def train_auc(d):
    """AUC on the training patients from export_preds.csv (B3 exports after training): the overfitting gap."""
    f = f"{d}/export_preds.csv"
    if not os.path.exists(f): return np.nan
    e = pd.read_csv(f); e = e[e.split == "train"]
    return stats.auc(e.label, e.prob)

def vo_train_auc(d):
    """Training-set AUC of a src.vitonly run (metrics.json): the overfitting gap."""
    return json.load(open(f"{d}/metrics.json"))["val"].get("train_auc", np.nan)

def compare(preds, ref):
    """One row per model on the patients all models share: AUC (DeLong CI), AUPRC, Brier, ECE, the 90%-spec
    operating point (threshold chosen on these same val patients: optimistic), and the paired DeLong test
    and paired bootstrap of the AUC difference vs `ref`."""
    preds = ev.align({k: v for k, v in preds.items() if v is not None})
    rows = []
    for k, d in preds.items():
        r = {"model": k, **ev.summary_row(d.label, d.prob)}
        if k != ref and ref in preds:
            y = preds[ref].label
            dl = ev.delong(y, preds[ref].prob, d.prob)
            pb = stats.paired_bootstrap_auc_diff(y, preds[ref].prob, d.prob)
            r.update(diff_vs_ref=dl["diff"], delong_ci_lo=dl["ci"][0], delong_ci_hi=dl["ci"][1], delong_p=dl["p"],
                     boot_p=pb["p"])
        rows.append(r)
    cols = ["model", "n", "pcr", "auc", "ci_lo", "ci_hi", "auprc", "brier", "ece", "sens@90spec", "ppv@90spec",
            "npv@90spec", "diff_vs_ref", "delong_ci_lo", "delong_ci_hi", "delong_p", "boot_p"]
    t = pd.DataFrame(rows)
    return t[[c for c in cols if c in t]]

def subtype_wide(preds, by="subtype"):
    """AUC (95% CI) per subtype (or cohort) for every model, all on the same patients; n / pCR+ per group."""
    preds = ev.align({k: v for k, v in preds.items() if v is not None})
    tabs = {k: ev.subgroup_table(d, by).set_index(by) for k, d in preds.items()}
    first = next(iter(tabs.values()))
    out = pd.DataFrame({"n": first.n.astype(int), "pCR+": first.pcr.astype(int)})
    for k, t in tabs.items():
        out[k] = [f"{a:.3f} ({lo:.2f}-{hi:.2f})" if a == a else "n/a (one class)"
                  for a, lo, hi in zip(t.auc, t.ci_lo, t.ci_hi)]
    return out

B1_VAL = repo_preds("b1_lr_s0_*")                       # clinical + tumour size, logistic regression (val 0.653)
B0_VAL = repo_preds("b0_lr_s0_*")                       # clinical only (val 0.639)
AKULA_B3_V2 = repo_preds("b3_vit_s0_1019-0302_*")       # Akula's B3 on cache v2 (val 0.559)
B3_V1 = repo_preds("b3_vit_s0_1009-1616_*")             # B3 on the old cache (val 0.530)
F2_V1 = repo_preds("f2_attn_s0_1009-1605_*")            # fusion on the old cache (val 0.690)
print("reference predictions from the repo:", {k: v is not None for k, v in
      dict(B1=B1_VAL, B0=B0_VAL, B3_v2_Akula=AKULA_B3_V2, B3_v1=B3_V1, F2_v1=F2_V1).items()})

In [ ]:
# ==== which cache is used for each cohort ====
from src.vitonly.slabs import find_caches as vo_find_caches, cache_label
VO_CACHES = vo_find_caches(["/kaggle/input", "/kaggle/working"], PREFER)
say(f"caches found: {len(VO_CACHES)} cohorts")
for c in COH:
    print(f"  {c:6s}", f"{cache_label(VO_CACHES[c])}  {VO_CACHES[c]}" if c in VO_CACHES else "MISSING -> left out", flush=True)
if not VO_CACHES:
    raise SystemExit("no cache attached: + Add Input -> mnj-cache2-* / mnj-cache-v2 / mnj-cache-*")
if any(cache_label(d).startswith("v1") for d in VO_CACHES.values()):
    print("NOTE: uint8 cache(s): the brightest enhancement is clipped, so intensity / PE features are underestimated. "
          "Raw mnj-cache2-* caches (step 2) are exact.", flush=True)

## 1. Probe: frozen features for every normalisation, per-slice vs volume twin

In [ ]:
PROBE_OUT = f"{OUT}/probe"
if RUN_PROBE:
    cmd = [sys.executable, "-m", "src.vitonly.probe", "--backbones", *PROBE_BACKBONES, "--norms", *PROBE_NORMS,
           "--crops", *PROBE_CROPS, "--prefer", PREFER, "--out", PROBE_OUT, "--cache-root", "/kaggle/input", "/kaggle/working"]
    if DEBUG: cmd += ["--limit", "40"]
    say("1. probe starts (progress every 100 patients)")
    run_cmd(cmd)
    PR = pd.read_csv(f"{PROBE_OUT}/probe_results.csv")
    PP = pd.read_csv(f"{PROBE_OUT}/probe_val_preds.csv", dtype={"pid": str})
    rows = []
    for (bb, crop, pool), g in PR[PR.backbone != "-"].groupby(["backbone", "crop", "pool"]):
        g = g.set_index("norm")
        for s, v in TWINS.items():
            if s in g.index and v in g.index:
                ks, kv = f"{bb}|{s}|{crop}|{pool}", f"{bb}|{v}|{crop}|{pool}"
                d = stats.paired_bootstrap_auc_diff(PP.label, PP[kv], PP[ks], n=1000)
                rows.append({"backbone": bb, "pool": pool, "crop": crop, "per_slice": s, "volume_twin": v,
                             "cv_twin": g.loc[v, "cv_auc"], "cv_slice": g.loc[s, "cv_auc"],
                             "cv_diff": g.loc[s, "cv_auc"] - g.loc[v, "cv_auc"],
                             "val_twin": g.loc[v, "val_auc"], "val_slice": g.loc[s, "val_auc"],
                             "val_diff": d["diff"], "val_diff_lo": d["ci"][0], "val_diff_hi": d["ci"][1], "p": d["p"]})
    TWIN_PROBE = pd.DataFrame(rows)
    TWIN_PROBE.to_csv(f"{OUT}/probe_per_slice_vs_volume.csv", index=False)
    print("per-slice minus volume twin (positive = per-slice better; CV on train is the low-noise number):")
    print(TWIN_PROBE.round(3).to_string(index=False))
    print("\ntop 10 probe rows by CV AUC:")
    print(PR.head(10)[["backbone", "norm", "crop", "pool", "cv_auc", "val_auc", "val_ci_lo", "val_ci_hi",
                       "val_auc_within"]].round(3).to_string(index=False))
else:
    say("1. probe skipped")

## 2. ComBat (+ subtype) on the ViT features

In [ ]:
DEEP_OUT = f"{OUT}/deep_combat"
if RUN_DEEP_COMBAT and os.path.isdir(PROBE_OUT):
    npz = sorted(glob.glob(f"{PROBE_OUT}/features_*.npz"))
    step9 = sorted(glob.glob("/kaggle/input/**/features_cohort/features_volume.csv", recursive=True))
    cmd = [sys.executable, "-m", "src.vitonly.features", "--out", DEEP_OUT, "--embeddings", *npz, "--prefer", PREFER,
           "--cache-root", "/kaggle/input", "/kaggle/working", "--jobs", "4"]
    if step9:
        say(f"2. step-9 features found ({step9[0]}): also scoring ViT + 3d+temporal")
        cmd += ["--features-from", os.path.dirname(step9[0]), "--refs", "volume", "--sets", "3d+temporal"]
    else:
        say("2. no step-9 output attached: ViT features only")
        cmd += ["--only-embeddings"]
    if DEBUG: cmd += ["--limit", "40"]
    run_cmd(cmd)
    DEEP = pd.read_csv(f"{DEEP_OUT}/results.csv")
    print(DEEP.head(20)[["ref", "set", "combat", "subtype", "cv_auc", "val_auc", "val_auc_within", "vs_lookup",
                         "vs_lookup_p"]].round(3).to_string(index=False))
else:
    say("2. skipped (RUN_DEEP_COMBAT = False or no probe output)")

## 3. Fine-tuning DINOv2 with per-slice vs volume normalisation

In [ ]:
def epoch_mean(d, lo=5, hi=15):
    """Mean val AUC over epochs lo..hi of a run (log.csv): less optimistic than the best epoch."""
    lg = pd.read_csv(f"{d}/log.csv")
    w = lg[(lg.epoch >= lo) & (lg.epoch <= hi)]
    return float(w.val_auc.mean()) if len(w) else float(lg.val_auc.mean())

FT_RUNS, FT_ROWS = {}, []
if RUN_FT:
    for norm in FT_NORMS:
        for seed in FT_SEEDS:
            if budget_left() < 0.6:
                say(f"3. time budget nearly used: skipping {norm} seed {seed}"); continue
            say(f"3. fine-tune {FT_CONFIG} norm={norm} seed={seed} ({len(FT_RUNS)} runs done)")
            try:
                d = finetune(FT_CONFIG, seed, [f"data.norm={norm}", *FT_EXTRA], label=f"ft_{norm}_s{seed}")
            except Exception as e:                       # one failed run must not lose the others
                say(f"3. FAILED {norm} seed {seed}: {type(e).__name__}: {e}"); continue
            m = json.load(open(f"{d}/metrics.json"))
            FT_RUNS.setdefault(norm, []).append(d)
            FT_ROWS.append({"norm": norm, "seed": seed, "best_epoch": m["best_epoch"], "val_auc_best": m["val"]["auc"],
                            "val_auc_ep5_15": epoch_mean(d), "train_auc": m["val"].get("train_auc", np.nan),
                            **{f"val_{c}": v["auc"] for c, v in m["val"]["by_cohort"].items()}})
            say(f"3. {norm} s{seed}: best-epoch val {FT_ROWS[-1]['val_auc_best']:.3f}, "
                f"epochs 5-15 mean {FT_ROWS[-1]['val_auc_ep5_15']:.3f}, train {FT_ROWS[-1]['train_auc']:.3f}")
            pd.DataFrame(FT_ROWS).to_csv(f"{OUT}/finetune_runs.csv", index=False)
            note_runs(f"ft_{norm}", FT_RUNS[norm])
    FT = pd.DataFrame(FT_ROWS)
    if len(FT):
        print(FT.round(3).to_string(index=False))
        print("\nmean over seeds:")
        print(FT.groupby("norm").mean(numeric_only=True).drop(columns="seed").round(3).to_string())
else:
    say("3. fine-tuning skipped")

In [ ]:
FT_PREDS, FT_CMP = {}, None
for norm, dirs in FT_RUNS.items():
    FT_PREDS[f"DINOv2 {norm} ({len(dirs)} seeds)"] = save_preds(f"ft_{norm}_ensemble", preds_of(dirs))
if FT_PREDS:
    FT_CMP = compare({"B1 LR (clinical + size)": B1_VAL, **FT_PREDS}, "B1 LR (clinical + size)")
    print(FT_CMP.round(3).to_string(index=False))
    rows = []
    for s, v in TWINS.items():
        ks = [k for k in FT_PREDS if f" {s} (" in k]; kv = [k for k in FT_PREDS if f" {v} (" in k]
        if ks and kv:
            a, b = ev.align({"v": FT_PREDS[kv[0]], "s": FT_PREDS[ks[0]]}).values()
            dl = ev.delong(a.label, a.prob, b.prob)
            rows.append({"per_slice": s, "volume_twin": v, "auc_twin": stats.auc(a.label, a.prob),
                         "auc_slice": stats.auc(b.label, b.prob), "diff": dl["diff"], "ci_lo": dl["ci"][0],
                         "ci_hi": dl["ci"][1], "delong_p": dl["p"]})
    FT_TWINS = pd.DataFrame(rows)
    print("\nfine-tuned, seed ensembles: per-slice minus volume twin (paired DeLong)")
    print(FT_TWINS.round(3).to_string(index=False))
    FT_TWINS.to_csv(f"{OUT}/finetune_per_slice_vs_volume.csv", index=False)
    print(subtype_wide(FT_PREDS, by="cohort").to_string())

## 4. Results file

In [ ]:
from src.vitonly.features import md_table
parts = [f"# Step 10 results: per-slice normalisation for the ViT, ComBat on ViT features ({time.strftime('%Y-%m-%d %H:%M')})", ""]
if RUN_PROBE and os.path.isdir(PROBE_OUT):
    parts += ["## Probe (frozen backbones): per-slice minus volume twin", "", md_table(TWIN_PROBE), ""]
if os.path.exists(f"{DEEP_OUT}/RESULTS.md"):
    parts += [open(f"{DEEP_OUT}/RESULTS.md").read().replace("# Step 9 results", "## ComBat on ViT features (from step 9 code)"), ""]
if FT_ROWS:
    parts += ["## Fine-tuned DINOv2 (2 blocks): every run", "", md_table(pd.DataFrame(FT_ROWS)), ""]
if FT_CMP is not None:
    parts += ["## Fine-tuned seed ensembles vs B1 (paired DeLong)", "", md_table(FT_CMP), "",
              "## Fine-tuned: per-slice minus volume twin", "", md_table(FT_TWINS), ""]
open(f"{OUT}/RESULTS_step10.md", "w").write("\n".join(parts) + "\n")
say(f"wrote {OUT}/RESULTS_step10.md")

In [ ]:
# ==== PUSH the small result files to the vaib branch (results/<PERSON>/...) ====
if PUSH_RESULTS and not _tok:
    print("PUSH_RESULTS needs the GITHUB_TOKEN secret; download results/ from the Output tab instead")
elif PUSH_RESULTS and not DEBUG:
    git("config", "user.name", f"Person {PERSON} (Kaggle)")
    git("config", "user.email", "kaggle@users.noreply.github.com")
    for attempt in range(3):
        git("fetch", "--quiet", "--depth", "1", "origin", f"+refs/heads/{GIT_TAG}:refs/remotes/origin/{GIT_TAG}")
        git("checkout", "--quiet", "-f", "-B", GIT_TAG, f"origin/{GIT_TAG}")
        shutil.copytree(f"/kaggle/working/results/{PERSON}", f"results/{PERSON}", dirs_exist_ok=True)
        git("add", f"results/{PERSON}")
        print(git("commit", "-m", f"results: {PERSON} {NOTEBOOK_NAME} ({GIT_TAG})"))
        try:
            git("push", "--quiet", "origin", GIT_TAG); print("pushed to", GIT_TAG); break
        except RuntimeError as e:
            print(f"push attempt {attempt + 1} failed:", e)
    else:
        print("NOT pushed: download results/ from the Output tab")